In [80]:
import sys
import pandas as pd
import numpy as np
from datetime import datetime
from geopy.geocoders import Nominatim
import pickle
import re

In [81]:
import certifi
import ssl
import geopy.geocoders
ctx = ssl.create_default_context(cafile=certifi.where())
geopy.geocoders.options.default_ssl_context = ctx

In [82]:
sys.path.append("../../code")
sys.path.append("../../analysis")
import loaders.cityprotect as cp

In [83]:
nom = Nominatim(user_agent="policedata", scheme='http')

In [84]:
previous_df = pd.read_pickle("../../data/clean_data/SCCSheriff_2017_2022_inferredCity.pkl")

In [85]:
dept = "Santa Clara County Office of the Sheriff"
df = cp.cityprotect("../../data/raw_data/{}".format(dept.replace(" ", "_")),
                                 start_date = previous_df["date"].max(),
                                  end_date = datetime.now() )

Starting: 2022-12-31 23:58:48
Ending:   2024-06-04 14:31:04.004939
Loading from 27 files.
Data file ../../data/raw_data/Santa_Clara_County_Office_of_the_Sheriff/Jan_Mar_2017_Santa_Clara_County_Office_of_the_Sheriff_report.csv is empty.
Data file ../../data/raw_data/Santa_Clara_County_Office_of_the_Sheriff/Apr_Jun_2017_Santa_Clara_County_Office_of_the_Sheriff_report.csv is empty.


In [87]:
len(df)

32065

In [88]:
print(df.head())

              ccn                date          updateDate                city  \
76329  S223650282 2022-12-31 23:58:48 2023-01-01 09:05:07  SANTA CLARA COUNTY   
76330  L223650085 2023-01-01 00:15:09 2023-01-01 09:05:06  SANTA CLARA COUNTY   
76331  L223650088 2023-01-01 00:36:17 2023-01-01 09:05:08  SANTA CLARA COUNTY   
76332  L223650094 2023-01-01 00:43:22 2023-01-01 09:05:06  SANTA CLARA COUNTY   
76333  L223650095 2023-01-01 00:45:05 2023-01-01 09:05:07  SANTA CLARA COUNTY   

      state postalCode          blocksizedAddress            incidentType  \
76329    CA          .    14000 Block LOMA RIO DR    PHONE UR OFFICE, OR:   
76330    CA          .         ALUM ROCK FALLS RD  SERVICE OR AID REQUEST   
76331    CA          .                  BONINO LN  SERVICE OR AID REQUEST   
76332    CA          .                      HWY 9  SERVICE OR AID REQUEST   
76333    CA          .  14300 Block LIDDICOAT CIR  SERVICE OR AID REQUEST   

      parentIncidentType                          

In [92]:
if "inferredCity" not in df.columns:
    df['inferredCity'] = ["unknown"]*len(df)

In [95]:
df.loc[df["inferredCity"].apply(lambda x: type(x) == float)] = "unknown"
df.loc[df["inferredCity"].apply(lambda x: x == 'blank')] = "unknown"
print(df["inferredCity"])

76329    unknown
76330    unknown
76331    unknown
76332    unknown
76333    unknown
          ...   
11614    unknown
11615    unknown
11616    unknown
11617    unknown
11618    unknown
Name: inferredCity, Length: 32065, dtype: object


In [98]:
# Some blocksizedAddress fields are NaNs, which is Not Helpful
df.loc[df["blocksizedAddress"].apply(lambda x: type(x) == float), "inferredCity"] = "blank"

# We will iterate over unique addresses, not records.
unique_addresses = df[df["inferredCity"] == "unknown"]["blocksizedAddress"].unique()
print("unique addresses", unique_addresses)

fail_count = 0

for idx, addr in enumerate(unique_addresses):
    print("{}: {}: {} instances".format(idx, addr, len(df[df["blocksizedAddress"] == addr])))
    result = cp.new_attempt_nom(nom, addr)
    #if not result:
        #result = attempt_goog(goog, addr)
        #googcount += 1

    if result:  # fix all the records at once
        df.loc[df["blocksizedAddress"] == addr, "inferredCity"] = result[0]
        df.loc[df["blocksizedAddress"] == addr, "postcode"] = result[1]
    else:
        print("Failed to identify {}".format(addr)) 
        fail_count += 1  

    #if idx%200 == 0:
        #print("Saving...")
        #df.to_pickle("SCCSheriff.pkl")
print("fail count:", fail_count, '/', len(unique_addresses))

unique addresses ['14000 Block LOMA RIO DR' 'ALUM ROCK FALLS RD' 'BONINO LN' ...
 '16200 Block CONDIT RD' 'PALM ST' '5400 Block WINFIELD BLVD']
0: 14000 Block LOMA RIO DR: 2 instances


AttributeError: 'function' object has no attribute 'new_attempt_nom'

In [ ]:
#df = cp.infer_cities(df, nom)